In [29]:
import re
import xml.etree.ElementTree as ET
import os
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re
import shutil
from sklearn.cluster import DBSCAN
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import math
import time
from collections import defaultdict
import hashlib
import torch
from sentence_transformers import SentenceTransformer
import xml.etree.ElementTree as ET
from xml.dom import minidom
import random
import xxhash
import gradio as gr
import html
import uuid
import json
import pandas as pd
from collections import defaultdict
import subprocess

In [30]:
from google.colab import drive

# mount the drive
drive.mount('/content/drive', force_remount=True)

# define the paths for the dataset
BASE_PATH = "/content/drive/MyDrive/PAN11/external-detection-corpus"

SOURCE_PATH = os.path.join(BASE_PATH, "source-document")
SUSPICIOUS_PATH = os.path.join(BASE_PATH, "suspicious-document")

print(f"Lookin for data in: {BASE_PATH}")

Mounted at /content/drive
Lookin for data in: /content/drive/MyDrive/PAN11/external-detection-corpus


In [31]:
SBERT_THRESHOLD = 0.60
EPS = 5000
MIN_SAMPLE = 2
MAX_OCC = 15

MAX_ANCHORS = 1000
FRAGMENT_WINDOW = 300
BATCH_SIZE = 1024
MODEL_NAME = 'paraphrase-multilingual-mpnet-base-v2'

In [32]:
%%writefile word_encoplot.cpp

#include <iostream>
#include <fstream>
#include <vector>
#include <string>
#include <algorithm>
#include <cctype>
#include <stdint.h>

struct Token {
    uint64_t hash;
    int offset;
};

struct NGram {
    uint64_t hash;
    int offset;
};

static uint64_t fnv_word(const std::string& s) {
    uint64_t h = 1469598103934665603ULL;
    for (unsigned char c : s) {
        h ^= c;
        h *= 1099511628211ULL;
    }
    return h;
}

static uint64_t fnv_combine(uint64_t h, uint64_t x) {
    h ^= x;
    h *= 1099511628211ULL;
    return h;
}

std::vector<Token> tokenize(const std::string& text) {
    std::vector<Token> result;
    std::string word;
    int start = -1;

    for (int i = 0; i < (int)text.size(); i++) {
        char c = text[i];

        if (c == ' ' || c == '\n' || c == '\t' || c == '\r' ||
            c == '.' || c == ','  || c == '!'  || c == '?'  ||
            c == ';' || c == ':'  || c == '('  || c == ')') {

            if (!word.empty()) {
                result.push_back({
                    fnv_word(word),
                    start
                });
                word.clear();
            }
            start = -1;
        }
        else {
            if (word.empty())
                start = i;
            if (c >= 'A' && c <= 'Z') {
                c = c + 32;
            }
            word.push_back(c);
        }
    }

    if (!word.empty()) {
        result.push_back({
            fnv_word(word),
            start
        });
    }

    return result;
}

std::vector<NGram> build_ngrams(const std::vector<Token>& tokens, int n) {
    std::vector<NGram> result;
    if ((int)tokens.size() < n)
        return result;

    for (int i = 0; i <= (int)tokens.size() - n; i++) {
        uint64_t h = 1469598103934665603ULL;
        for (int j = 0; j < n; j++) {
            h = fnv_combine(h, tokens[i+j].hash);
        }
        result.push_back({
            h,
            tokens[i].offset
        });
    }
    return result;
}

std::string read_file(const char* path) {
    std::ifstream f(path);
    if (!f.good())
        return "";

    return std::string(
        std::istreambuf_iterator<char>(f),
        std::istreambuf_iterator<char>()
    );
}

int main(int argc, char** argv) {
    if (argc != 3)
        return 1;

    std::string susp_text = read_file(argv[1]);
    std::string src_text  = read_file(argv[2]);

    auto susp_tokens = tokenize(susp_text);
    auto src_tokens  = tokenize(src_text);

    auto susp_ngrams = build_ngrams(susp_tokens, 4);
    auto src_ngrams  = build_ngrams(src_tokens, 4);

    std::sort(susp_ngrams.begin(), susp_ngrams.end(), [](const NGram& a, const NGram& b) {
        return a.hash < b.hash;
    });

    std::sort(src_ngrams.begin(), src_ngrams.end(), [](const NGram& a, const NGram& b) {
        return a.hash < b.hash;
    });

    const int MAX_OCC = 15;

    size_t i = 0;
    size_t j = 0;

    while (i < susp_ngrams.size() && j < src_ngrams.size()) {
        uint64_t h1 = susp_ngrams[i].hash;
        uint64_t h2 = src_ngrams[j].hash;

        if (h1 == h2) {
            size_t i_start = i;
            size_t j_start = j;

            while (i < susp_ngrams.size() && susp_ngrams[i].hash == h1) {
                i++;
            }

            while (j < src_ngrams.size() && src_ngrams[j].hash == h2) {
                j++;
            }

            size_t susp_count = i - i_start;
            size_t src_count  = j - j_start;

            if (susp_count <= MAX_OCC && src_count <= MAX_OCC) {
                for (size_t a = i_start; a < i; a++) {
                    for (size_t b = j_start; b < j; b++) {
                        std::cout << susp_ngrams[a].offset << " " << src_ngrams[b].offset << "\n";
                    }
                }
            }
        }
        else if (h1 < h2) {
            while (i < susp_ngrams.size() && susp_ngrams[i].hash == h1) {
                i++;
            }
        }
        else {
            while (j < src_ngrams.size() && src_ngrams[j].hash == h2) {
                j++;
            }
        }
    }

    return 0;
}

Overwriting word_encoplot.cpp


In [33]:
!g++ -O3 -march=native word_encoplot.cpp -o word_encoplot

In [34]:
class Document:
  def __init__(self, doc_id, is_source=True, text_content=None):
    self.numeric_id = int(doc_id)
    self.is_source = is_source

    if text_content is not None:
      self.text = text_content
      self.doc_name = f"demo_doc_{doc_id}.txt"
      self.xml_path = None
      self.file_path = None
      self.metadata = {}
      self.language = 'english'
    else:
      formatted_id = f"{self.numeric_id:05d}"
      prefix = "source-document" if is_source else "suspicious-document"

      part_number = ((self.numeric_id - 1) // 500) + 1
      part_folder = f"part{part_number}"
      base_folder = SOURCE_PATH if is_source else SUSPICIOUS_PATH

      self.doc_name = f"{prefix}{formatted_id}.txt"
      self.xml_name = f"{prefix}{formatted_id}.xml"

      self.file_path = os.path.join(base_folder, part_folder, self.doc_name)
      self.xml_path = os.path.join(base_folder, part_folder, self.xml_name)

      with open(self.file_path, "r", encoding='utf-8', errors='ignore') as f:
        self.text = f.read()

      self.metadata = self._parse_xml()
      self.language = self.metadata.get('lang', 'english')

    self.segments = []

  def _parse_xml(self):
    if self.xml_path is None:
        return {}
    meta = {}
    self.plagiarism_features = []
    try:
        tree = ET.parse(self.xml_path)
        root = tree.getroot()

        for feature in root.findall('feature'):
            if feature.get('name') == 'about':
                meta['lang'] = feature.get('lang', 'en')

            if feature.get('name') == 'md5Hash':
                meta['md5'] = feature.get('value')

            if feature.get('name') == 'plagiarism':
                self.plagiarism_features.append(PlagiarismFeature(
                    this_offset=int(feature.get('this_offset')),
                    this_length=int(feature.get('this_length')),
                    source_reference=feature.get('source_reference'),
                    source_offset=int(feature.get('source_offset')),
                    source_length=int(feature.get('source_length')),
                    obfuscation=feature.get('obfuscation', 'none')
                ))
    except Exception as e:
        print(f"Error parsing XML: {e}")
    return meta

In [35]:
class PlagiarismFeature:
    def __init__(self, this_offset, this_length, source_reference,
                 source_offset, source_length, obfuscation):
        self.this_offset = this_offset
        self.this_length = this_length
        self.source_reference = source_reference
        self.source_offset = source_offset
        self.source_length = source_length
        self.obfuscation = obfuscation

    def get_source_id(self):
        return int(self.source_reference
                       .replace("source-document", "")
                       .replace(".txt", ""))

    def __repr__(self):
        return (f"PlagiarismFeature("
                f"offset={self.this_offset}, "
                f"length={self.this_length}, "
                f"source={self.source_reference}, "
                f"obfuscation={self.obfuscation})")

In [36]:
class PredictedSegment:
    def __init__(self, susp_id, src_id, susp_off, susp_len, src_off, src_len, score):
        self.susp_id = susp_id
        self.src_id = src_id
        self.susp_off = susp_off
        self.susp_len = susp_len
        self.src_off = src_off
        self.src_len = src_len
        self.score = score

    def __repr__(self):
        return f"<Match Susp:{self.susp_id} Src:{self.src_id} Score:{self.score:.2f}>"

In [37]:
class WordEncoplotEngine:

    def __init__(self, executable="./word_encoplot", max_anchors=MAX_ANCHORS):
        self.executable = executable
        self.max_anchors = max_anchors

    def compute_document_hashes(self, document):
        return None

    def get_anchors_for_pair(self, susp_doc, src_doc, susp_hashes=None, src_hashes=None):
        result = subprocess.run([self.executable, susp_doc.file_path, src_doc.file_path], capture_output=True, text=True)

        anchors = []

        for line in result.stdout.splitlines():

            parts = line.strip().split()

            if len(parts) != 2:
                continue

            anchors.append([int(parts[0]), int(parts[1])])

        anchors = self.apply_bucketing(anchors, len(susp_doc.text))

        anchors.sort(key=lambda x: x[0])

        return anchors

    def run_mass_scan(self, susp_path, source_paths):
        anchors_by_source = {path: [] for path in source_paths}

        for src_path in source_paths:
            result = subprocess.run([self.executable, susp_path, src_path], capture_output=True, text=True)

            anchors = []
            for line in result.stdout.splitlines():
                parts = line.strip().split()
                if len(parts) != 2:
                    continue
                anchors.append([int(parts[0]), int(parts[1])])

            anchors.sort(key=lambda x: x[0])
            anchors_by_source[src_path] = anchors

        return anchors_by_source

    def apply_bucketing(self, anchors, doc_len):
        if len(anchors) <= self.max_anchors:
            return anchors

        bucket_size = max(1, doc_len // self.max_anchors)

        buckets = {}

        for p_susp, p_src in anchors:

            idx = p_susp // bucket_size

            if idx not in buckets:
                buckets[idx] = (p_susp, p_src)

        return list(buckets.values())

In [38]:
engine = WordEncoplotEngine()

In [40]:
class SemanticAnalyzer:
  def __init__(self, model_name=MODEL_NAME, progress=None):
    self.device = "cuda" if torch.cuda.is_available() else "cpu"

    if progress is not None:
        progress(0.05, desc=f"Loading SBERT model ({model_name}) into memory...")

    print(f"[INIT] Loading {model_name}...")
    self.model = SentenceTransformer(model_name)

    if self.device == "cuda":
        print("[INIT] Converting model to FP16 (Mixed Precision)")
        self.model = self.model.half()

    self.model = self.model.to(self.device)

  def verify_anchors(self, susp_doc, src_doc, anchors, progress=None):
    return self.verify_multiple(susp_doc, [(src_doc, anchors)], progress)

  def verify_multiple(self, susp_doc, source_anchor_pairs, progress=None):
    all_texts_susp = []
    all_texts_src = []
    all_meta = []

    for src_doc, anchors in source_anchor_pairs:
        for s_off, src_off in anchors:
            f_susp, s_off_actual, s_len_actual = get_safe_fragment(susp_doc.text, s_off, FRAGMENT_WINDOW)
            f_src, src_off_actual, src_len_actual = get_safe_fragment(src_doc.text, src_off, FRAGMENT_WINDOW)

            if not f_susp or not f_src:
                continue

            t_susp = re.sub(r'\s+', ' ', f_susp).strip().lower()
            t_src  = re.sub(r'\s+', ' ', f_src).strip().lower()

            if len(t_susp) > 30:
                all_texts_susp.append(t_susp)
                all_texts_src.append(t_src)
                all_meta.append({
                    'susp_id': susp_doc.numeric_id,
                    'src_id': src_doc.numeric_id,
                    's_off': s_off_actual, 's_len': s_len_actual,
                    'src_off': src_off_actual, 'src_len': src_len_actual,
                })

    if not all_texts_susp:
        return [], [], np.array([])

    if progress is not None:
        emb_susp_list = []
        emb_src_list = []
        num_chunks = math.ceil(len(all_texts_susp) / BATCH_SIZE)

        for i in progress.tqdm(range(num_chunks), desc=f"SBERT Encoding {len(all_texts_susp)} Pairs"):
            start_idx = i * BATCH_SIZE
            end_idx = min((i + 1) * BATCH_SIZE, len(all_texts_susp))

            chunk_susp = all_texts_susp[start_idx:end_idx]
            chunk_src = all_texts_src[start_idx:end_idx]

            e_s = self.model.encode(chunk_susp, convert_to_numpy=True, convert_to_tensor=False, show_progress_bar=False, batch_size=BATCH_SIZE)
            e_r = self.model.encode(chunk_src, convert_to_numpy=True, convert_to_tensor=False, show_progress_bar=False, batch_size=BATCH_SIZE)

            emb_susp_list.append(e_s)
            emb_src_list.append(e_r)

        emb_susp = np.vstack(emb_susp_list)
        emb_src = np.vstack(emb_src_list)
    else:
        emb_susp = self.model.encode(all_texts_susp, convert_to_numpy=True, convert_to_tensor=False, show_progress_bar=False, batch_size=BATCH_SIZE)
        emb_src  = self.model.encode(all_texts_src,  convert_to_numpy=True, convert_to_tensor=False, show_progress_bar=False, batch_size=BATCH_SIZE)

    scores = np.zeros(len(all_texts_susp), dtype=np.float32)
    chunk_size = 50000
    for i in range(0, len(all_texts_susp), chunk_size):
        j = min(i + chunk_size, len(all_texts_susp))
        dot = np.sum(emb_susp[i:j] * emb_src[i:j], axis=1)
        n_s = np.linalg.norm(emb_susp[i:j], axis=1)
        n_src = np.linalg.norm(emb_src[i:j], axis=1)
        scores[i:j] = dot / (n_s * n_src + 1e-8)

    del emb_susp
    del emb_src
    torch.cuda.empty_cache()

    verified_fragments = []
    for i, score in enumerate(scores):
        if score >= SBERT_THRESHOLD:
            m = all_meta[i]
            verified_fragments.append(
                PredictedSegment(m['susp_id'], m['src_id'], m['s_off'], m['s_len'], m['src_off'], m['src_len'], score)
            )
    return verified_fragments, all_meta, scores

In [41]:
class PlagiarismCase:
    def __init__(self, susp_id, susp_off, susp_len, src_id, src_off, src_len, score):
        self.susp_id  = susp_id
        self.susp_off = susp_off
        self.susp_len = susp_len
        self.src_id   = src_id
        self.src_off  = src_off
        self.src_len  = src_len
        self.score    = score

    def __repr__(self):
        return (f"PlagiarismCase(susp={self.susp_id}, src={self.src_id}, "
                f"susp_off={self.susp_off}, src_off={self.src_off}, "
                f"score={self.score:.2f})")

In [42]:
class ValidationMetrics:
    def __init__(self):
        pass

    def compute_granularity(self, gt_fragments, detected_fragments):
        if not gt_fragments:
            return 1.0
        total_gran = 0
        for gt_start, gt_end in gt_fragments:
            overlapping = 0
            for det_start, det_end in detected_fragments:
                if det_start < gt_end and det_end > gt_start:
                    overlapping += 1
            total_gran += max(1, overlapping)
        return total_gran / len(gt_fragments)

    def calculate_metrics(self, ground_truth_features, predicted_segments):
        metrics = {
            "recall": 0.0,
            "precision": 0.0,
            "f1": 0.0,
            "granularity": 1.0,
            "plagdet": 0.0
        }

        if not ground_truth_features:
            metrics["recall"] = 1.0
            return metrics

        # recall
        total_recall = 0
        for gt in ground_truth_features:
            covered_len = 0
            for pred in predicted_segments:
                intersect_start = max(gt.this_offset, pred.susp_off)
                intersect_end = min(gt.this_offset + gt.this_length, pred.susp_off + pred.susp_len)
                if intersect_end > intersect_start:
                    covered_len += (intersect_end - intersect_start)
            total_recall += (covered_len / gt.this_length)
        metrics["recall"] = total_recall / len(ground_truth_features)

        # precision
        if predicted_segments:
            total_precision = 0
            for pred in predicted_segments:
                covered_len = 0
                for gt in ground_truth_features:
                    intersect_start = max(gt.this_offset, pred.susp_off)
                    intersect_end = min(gt.this_offset + gt.this_length, pred.susp_off + pred.susp_len)
                    if intersect_end > intersect_start:
                        covered_len += (intersect_end - intersect_start)
                total_precision += (covered_len / pred.susp_len)
            metrics["precision"] = total_precision / len(predicted_segments)

        # granularity
        gt_tuples = [(f.this_offset, f.this_offset + f.this_length) for f in ground_truth_features]
        det_tuples = [(d.susp_off, d.susp_off + d.susp_len) for d in predicted_segments]
        metrics["granularity"] = self.compute_granularity(gt_tuples, det_tuples)

        # f1 and plagDet
        if metrics["precision"] + metrics["recall"] > 0:
            metrics["f1"] = 2 * (metrics["precision"] * metrics["recall"]) / (metrics["precision"] + metrics["recall"])

        metrics["plagdet"] = metrics["f1"] / math.log2(1 + metrics["granularity"]) if metrics["granularity"] > 0 else 0

        return {k: round(v, 4) for k, v in metrics.items()}


    def calculate_global_score(self, all_results_list):
      relevant_results = [res for res in all_results_list if not (res['recall'] == 1.0 and res['precision'] == 0.0)]

      if not relevant_results:
        print("\n" + "!"*60)
        print("  NO RELEVANT DATA TO CALCULATE GLOBAL SCORE")
        print("  (All documents were original and correctly identified as such)")
        print("!"*60)
        return None

      n = len(relevant_results)
      avg_metrics = {k: sum(res[k] for res in relevant_results) / n for k in relevant_results[0].keys()}

      print("\n" + "="*60)
      print(f"GLOBAL SCORE (average for {n} relevant documents)")
      print("-" * 60)
      print(f"  Precision    : {avg_metrics['precision']:.4f}")
      print(f"  Recall       : {avg_metrics['recall']:.4f}")
      print(f"  F1-Score     : {avg_metrics['f1']:.4f}")
      print(f"  Granularity  : {avg_metrics['granularity']:.4f}")
      print(f"  PlagDet      : {avg_metrics['plagdet']:.4f}")
      print("="*60 + "\n")

      return avg_metrics

    def analyze_by_obfuscation(self, suspicious_docs, detections):
        import math

        obf_data = {}

        for doc in suspicious_docs:
            susp_id = doc.numeric_id
            my_dets = detections.get(susp_id, [])

            for gt in doc.plagiarism_features:
                obf_type = gt.obfuscation if gt.obfuscation else "unknown"

                if obf_type not in obf_data:
                    obf_data[obf_type] = {
                        'gt_count': 0, 'det_count': 0,
                        'gt_fragments': [], 'det_fragments': [],
                        'gt_chars': set(), 'det_chars': set()
                    }

                obf_data[obf_type]['gt_count'] += 1
                gt_start, gt_end = gt.this_offset, gt.this_offset + gt.this_length
                obf_data[obf_type]['gt_fragments'].append((gt_start, gt_end))

                for c in range(gt_start, gt_end):
                    obf_data[obf_type]['gt_chars'].add((susp_id, c))

            for det in my_dets:
                det_start, det_end = det.susp_off, det.susp_off + det.susp_len
                assigned_obf = None
                max_overlap = 0

                for gt in doc.plagiarism_features:
                    overlap = max(0, min(gt.this_offset + gt.this_length, det_end) - max(gt.this_offset, det_start))
                    if overlap > max_overlap:
                        max_overlap = overlap
                        assigned_obf = gt.obfuscation if gt.obfuscation else "unknown"

                if assigned_obf is None:
                    assigned_obf = "false_positive_only"

                if assigned_obf not in obf_data:
                    obf_data[assigned_obf] = {
                        'gt_count': 0, 'det_count': 0,
                        'gt_fragments': [], 'det_fragments': [],
                        'gt_chars': set(), 'det_chars': set()
                    }

                obf_data[assigned_obf]['det_count'] += 1
                obf_data[assigned_obf]['det_fragments'].append((det_start, det_end))
                for c in range(det_start, det_end):
                    obf_data[assigned_obf]['det_chars'].add((susp_id, c))

        print("\n" + "="*85)
        print("DEEP PERFORMANCE ANALYSIS BY PLAGIARISM OBFUSCATION TYPE")
        print("-" * 85)
        print(f"{'Obfuscation Type':<18} | {'GT/Det':<8} | {'Precision':<10} | {'Recall':<10} | {'F1-Score':<10} | {'Gran':<6} | {'PlagDet':<8}")
        print("-" * 85)

        for obf_type, data in obf_data.items():
            if obf_type == "false_positive_only":
                continue

            gt_chars = data['gt_chars']
            det_chars = data['det_chars']

            tp_chars = len(gt_chars & det_chars)

            precision = tp_chars / len(det_chars) if len(det_chars) > 0 else 0.0
            recall = tp_chars / len(gt_chars) if len(gt_chars) > 0 else 0.0

            f1 = 0.0
            if precision + recall > 0:
                f1 = 2 * (precision * recall) / (precision + recall)

            gran = self.compute_granularity(data['gt_fragments'], data['det_fragments'])

            plagdet = f1 / math.log2(1 + gran) if gran > 0 else 0.0

            print(f"{obf_type:<18} | {data['gt_count']}/{data['det_count']:<6} | {precision:.4f}    | {recall:.4f} | {f1:.4f}   | {gran:.2f} | {plagdet:.4f}")

        print("="*85 + "\n")


In [43]:
def get_safe_fragment(text, offset, win=FRAGMENT_WINDOW):
        if not text:
            return "", 0, 0

        text_len = len(text)
        start = max(0, min(offset, text_len - 1))

        while start > 0:
            if text[start-1] in [' ', '\n', '\t']:
                break
            start -= 1

        end = min(text_len, start + win)

        while end < len(text) and text[end] not in [' ', '\n', '.', '!', '?']:
            end += 1

        fragment = text[start:end]
        return fragment, start, end - start

In [44]:
def extract_sample(min_susp_id, max_susp_id):
    suspicious_docs = []
    source_ids_needed = set()

    for i in range(min_susp_id, max_susp_id + 1):
        try:
            doc = Document(i, is_source=False)
            suspicious_docs.append(doc)

            for pf in doc.plagiarism_features:
                src_id = int(pf.source_reference
                               .replace("source-document", "")
                               .replace(".txt", ""))
                source_ids_needed.add(src_id)

        except Exception as e:
            print(f"[!] Suspicious {i:05d} error: {e}")
            continue

    print(f"Loaded suspicious docs : {len(suspicious_docs)}")
    print(f"Unique sources : {len(source_ids_needed)}")

    return suspicious_docs, sorted(list(source_ids_needed))

In [46]:
def extract_sources_for_suspicious(susp_id, n_sources, source_pool_size=11093):
    doc = Document(susp_id, is_source=False)

    required_ids = set()
    for pf in doc.plagiarism_features:
        src_id = int(pf.source_reference
                       .replace("source-document", "")
                       .replace(".txt", ""))
        required_ids.add(src_id)

    print(f"[SOURCES] Doc {susp_id:05d} | "
          f"GT sources: {len(required_ids)} | "
          f"Target total: {n_sources}")

    if len(required_ids) >= n_sources:
        print(f"[SOURCES] GT sources ({len(required_ids)}) >= n_sources ({n_sources}), returning all GT sources.")
        return doc, sorted(required_ids)

    all_possible = set(range(1, source_pool_size + 1)) - required_ids
    n_random     = n_sources - len(required_ids)
    random_ids   = set(random.sample(sorted(all_possible), n_random))

    final_ids = sorted(required_ids | random_ids)

    print(f"[SOURCES] GT: {sorted(required_ids)} | "
          f"Random fill: {n_random} | "
          f"Total: {len(final_ids)}")

    return doc, final_ids

In [47]:
def merge_plagiarism_cases(cases, proximity=0):
    if not cases:
        return []

    cases.sort(key=lambda x: x.susp_off)

    merged_cases = []
    current_case = cases[0]

    for i in range(1, len(cases)):
        next_case = cases[i]

        if next_case.susp_off <= (current_case.susp_off + current_case.susp_len + proximity):
            new_susp_end = max(current_case.susp_off + current_case.susp_len, next_case.susp_off + next_case.susp_len)
            current_case.susp_len = new_susp_end - current_case.susp_off

            new_src_end = max(current_case.src_off + current_case.src_len, next_case.src_off + next_case.src_len)
            current_case.src_len = new_src_end - current_case.src_off

            current_case.score = max(current_case.score, next_case.score)
        else:
            merged_cases.append(current_case)
            current_case = next_case

    merged_cases.append(current_case)
    return merged_cases

def cluster_results_dbscan(hits, eps=EPS, min_samples=MIN_SAMPLE):
    if not hits:
        return []

    hits_by_source = defaultdict(list)
    for h in hits:
        hits_by_source[h.src_id].append(h)

    final_results = []

    for src_id, src_hits in hits_by_source.items():
        points = np.array([[h.susp_off, h.src_off] for h in src_hits])
        db     = DBSCAN(eps=eps, min_samples=min_samples).fit(points)
        labels = db.labels_

        clustered_cases_for_src = []

        for label in set(labels):
            if label == -1:
                continue

            cluster = [src_hits[i] for i, l in enumerate(labels) if l == label]

            susp_min  = min(h.susp_off for h in cluster)
            susp_max  = max(h.susp_off + h.susp_len for h in cluster)
            src_min   = min(h.src_off for h in cluster)
            src_max   = max(h.src_off + h.src_len for h in cluster)
            avg_score = sum(h.score for h in cluster) / len(cluster)

            clustered_cases_for_src.append(PlagiarismCase(
                susp_id  = cluster[0].susp_id,
                susp_off = susp_min,
                susp_len = susp_max - susp_min,
                src_id   = src_id,
                src_off  = src_min,
                src_len  = src_max - src_min,
                score    = avg_score
            ))

        merged_cases_for_src = merge_plagiarism_cases(clustered_cases_for_src, proximity=eps)
        final_results.extend(merged_cases_for_src)

    return final_results

In [48]:
analyzer = SemanticAnalyzer()
validator = ValidationMetrics()

[INIT] Loading paraphrase-multilingual-mpnet-base-v2...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[INIT] Converting model to FP16 (Mixed Precision)


In [50]:
def run_pipeline(min_susp_id, max_susp_id, analyzer=None):
    if analyzer is None:
        if 'analyzer' in globals():
            analyzer = globals()['analyzer']
            print("[INFO] Using existing analyzer found in memory.")
        else:
            print("[INFO] Initializing new SemanticAnalyzer...")
            analyzer = SemanticAnalyzer()

    print("-" * 80)
    print(f"[STEP 1] Extracting sample: suspicious files from {min_susp_id} to {max_susp_id}")
    print("-" * 80)
    suspicious_docs, source_ids = extract_sample(min_susp_id, max_susp_id)
    print(f"Necessary sources: {len(source_ids)}\n")

    print("[STEP 2] Loading source docs...")
    source_docs = []
    for s_id in source_ids:
        try:
            source_docs.append(Document(s_id, is_source=True))
        except Exception as e:
            print(f"  [!] Error loading source {s_id}: {e}")
    print(f"Sources loaded: {len(source_docs)}\n")

    print("[STEP 3] Lexical processinf with FVN-1A")
    t3 = time.time()

    all_texts_susp = []
    all_texts_src = []
    all_meta = []

    for doc_susp in suspicious_docs:
        src_paths = [d.file_path for d in source_docs]
        raw_results = engine.run_mass_scan(doc_susp.file_path, src_paths)

        for s_path, anchors in raw_results.items():
            if not anchors: continue

            raw_anchor_count = len(anchors)
            anchors = engine.apply_bucketing(anchors, len(doc_susp.text))
            s_id = int(re.search(r'document(\d+)', s_path).group(1))

            for p_susp, p_src in anchors:
                f_susp, s_off, s_len = get_safe_fragment(doc_susp.text, p_susp)
                doc_src = next((d for d in source_docs if d.numeric_id == s_id), None)
                if not doc_src: continue
                f_src, src_off, src_len = get_safe_fragment(doc_src.text, p_src)

                if not f_susp or not f_src: continue

                t_susp = re.sub(r'\s+', ' ', f_susp).strip().lower()
                t_src  = re.sub(r'\s+', ' ', f_src).strip().lower()

                if len(t_susp) > 30:
                    all_texts_susp.append(t_susp)
                    all_texts_src.append(t_src)
                    all_meta.append({
                        'susp_id': doc_susp.numeric_id, 'src_id': s_id,
                        's_off': s_off, 's_len': s_len,
                        'src_off': src_off, 'src_len': src_len,
                        'raw_anchors': raw_anchor_count
                    })

    t3_elapsed = time.time() - t3
    print(f"Candidate fragments from N-gram Engine: {len(all_texts_susp)} | Time: {t3_elapsed/60:.1f}min\n")

    print(f"[STEP 4] SBERT batch encoding for {len(all_texts_susp)} pairs...")
    t5 = time.time()

    emb_susp = analyzer.model.encode(all_texts_susp, convert_to_numpy=True, convert_to_tensor=False, show_progress_bar=True, batch_size=BATCH_SIZE)
    emb_src  = analyzer.model.encode(all_texts_src,  convert_to_numpy=True, convert_to_tensor=False, show_progress_bar=True, batch_size=BATCH_SIZE)

    print("[INFO] Computing cosine similarities in chunks")
    scores = np.zeros(len(all_texts_susp), dtype=np.float32)
    chunk_size = 50000
    for i in range(0, len(all_texts_susp), chunk_size):
        j = min(i + chunk_size, len(all_texts_susp))
        dot = np.sum(emb_susp[i:j] * emb_src[i:j], axis=1)
        n_s = np.linalg.norm(emb_susp[i:j], axis=1)
        n_src = np.linalg.norm(emb_src[i:j], axis=1)
        scores[i:j] = dot / (n_s * n_src + 1e-8)

    del emb_susp
    del emb_src
    torch.cuda.empty_cache()

    print(f"Encoding done in {time.time() - t5:.1f}s\n")

    print("[STEP 5] Filtering and clustering with DBSCAN")
    hits_per_pair = defaultdict(list)
    for i, score in enumerate(scores):
        if score >= SBERT_THRESHOLD:
            m = all_meta[i]
            hits_per_pair[(m['susp_id'], m['src_id'])].append(
                PredictedSegment(m['susp_id'], m['src_id'], m['s_off'], m['s_len'], m['src_off'], m['src_len'], score)
            )

    detections = defaultdict(list)
    for (susp_id, src_id), hits in hits_per_pair.items():
        clustered = cluster_results_dbscan(hits)
        detections[susp_id].extend(clustered)
    print(f"Pairs with detected plagiarism: {len(hits_per_pair)}\n")

    print("[STEP 6] Generating XML output...")
    os.makedirs("/content/output", exist_ok=True)
    for doc_susp in suspicious_docs:
        root = ET.Element("document", reference=doc_susp.doc_name)
        for det in detections.get(doc_susp.numeric_id, []):
            ET.SubElement(root, "feature",
                name="detected-plagiarism",
                this_offset=str(det.susp_off), this_length=str(det.susp_len),
                source_reference=f"source-document{det.src_id:05d}.txt",
                source_offset=str(det.src_off), source_length=str(det.src_len)
            )
        ET.ElementTree(root).write(f"/content/output/{doc_susp.doc_name.replace('.txt', '.xml')}", encoding='utf-8', xml_declaration=True)

    all_results = []
    print("[STEP 7] Detected vs ground truth...")
    print("-" * 60)

    for doc_susp in suspicious_docs:
        m = validator.calculate_metrics(doc_susp.plagiarism_features, detections.get(doc_susp.numeric_id, []))
        all_results.append(m)
        if len(doc_susp.plagiarism_features) > 0 or len(detections.get(doc_susp.numeric_id, [])) > 0:
            print(f"  Suspicious {doc_susp.numeric_id:05d} | GT: {len(doc_susp.plagiarism_features)} | "
                  f"Det: {len(detections.get(doc_susp.numeric_id, []))} | P: {m['precision']:.2f} R: {m['recall']:.2f}")

    validator.calculate_global_score(all_results)

    print("\n[STEP 9] Running obfuscation analysis...")
    validator.analyze_by_obfuscation(suspicious_docs, detections)

    return suspicious_docs, source_docs, detections, scores, all_meta

In [51]:
import nbformat

with open('/content/drive/MyDrive/Colab Notebooks/licenta.ipynb', 'r') as f:
    nb = nbformat.read(f, as_version=4)

if 'widgets' in nb.metadata:
    del nb.metadata['widgets']

for cell in nb.cells:
    if 'metadata' in cell:
        if 'executionInfo' in cell.metadata:
            del cell.metadata['executionInfo']

with open('/content/drive/MyDrive/Colab Notebooks/licenta_clean.ipynb', 'w') as f:
    nbformat.write(nb, f)

print("Done!")

Done!


In [52]:
import matplotlib.pyplot as plt

CASE_COLORS = [
    "#ffb3ba",
    "#ffdfba",
    "#ffffba",
    "#baffc9",
    "#bae1ff",
    "#d5baff",
    "#ffd6e0",
    "#c9f9ff"
]

def escape_html_keep_newlines(text):
    text = html.escape(text)
    return text.replace("\n", "<br>")

def build_highlighted_html(text, plagiarism_cases):
    if not plagiarism_cases:
        return f"""
        <div style="padding:20px; line-height:1.6;">
            {escape_html_keep_newlines(text)}
        </div>
        """

    spans = []

    for idx, case in enumerate(plagiarism_cases):
        color = CASE_COLORS[idx % len(CASE_COLORS)]

        spans.append({
            "start": case.susp_off,
            "end": case.susp_off + case.susp_len,
            "color": color,
            "case_id": idx,
            "case": case
        })

    spans.sort(key=lambda x: x["start"])

    html_parts = []
    current_pos = 0

    for span in spans:
        start = span["start"]
        end = span["end"]

        if current_pos < start:
            normal_text = text[current_pos:start]
            html_parts.append(
                escape_html_keep_newlines(normal_text)
            )

        fragment = text[start:end]

        tooltip = f"""
        CASE #{span['case_id']}

        Source offset: {span['case'].src_off}
        Source length: {span['case'].src_len}

        Suspicious offset: {span['case'].susp_off}
        Suspicious length: {span['case'].susp_len}

        Similarity: {span['case'].score:.2f}
        """

        highlighted = f"""
        <span
            id="case-{span['case_id']}"
            title="{tooltip}"
            style="
                background-color:{span['color']};
                padding:2px;
                border-radius:4px;
                cursor:pointer;
            "
        >
            {escape_html_keep_newlines(fragment)}
        </span>
        """

        html_parts.append(highlighted)
        current_pos = end

    if current_pos < len(text):
        html_parts.append(
            escape_html_keep_newlines(text[current_pos:])
        )

    return f"""
    <div style="
        padding:20px;
        line-height:1.8;
        font-family:Arial;
        white-space:pre-wrap;
    ">
        {''.join(html_parts)}
    </div>
    """

def build_case_dataframe(plagiarism_cases, suspicious_text, source_text):
    rows = []
    for idx, case in enumerate(plagiarism_cases):
        susp_fragment = suspicious_text[case.susp_off : case.susp_off + case.susp_len]
        if source_text is not None:
            source_fragment = source_text[case.src_off : case.src_off + case.src_len][:500]
        else:
            src_doc_obj = Document(case.src_id, is_source=True)
            source_fragment = src_doc_obj.text[case.src_off : case.src_off + case.src_len][:500]

        rows.append({
            "Case": idx,
            "Source ID": f"{case.src_id:05d}",
            "Similarity": round(float(case.score), 2),
            "Suspicious offset": case.susp_off,
            "Suspicious length": case.susp_len,
            "Source offset": case.src_off,
            "Source length": case.src_len,
            "Suspicious fragment": susp_fragment[:500],
            "Source fragment": source_fragment
        })

    return pd.DataFrame(rows)

def plot_validation_timeline(susp_doc, plagiarism_cases, case_colors):
    fig, ax = plt.subplots(figsize=(15, 3))

    gt_intervals = []
    for pf in susp_doc.plagiarism_features:
        start = pf.this_offset
        end = start + pf.this_length
        gt_intervals.append((start, end - start))

    if gt_intervals:
        ax.broken_barh(gt_intervals, (10, 5), facecolors='tab:gray', edgecolors='white', label='Ground Truth')
        for start, length in gt_intervals:
            ax.text(start, 12.5, f"{start}", va='center', ha='right', fontsize=9)
            ax.text(start + length, 12.5, f"{start + length}", va='center', ha='left', fontsize=9)

    pred_intervals = []
    colors = []
    for idx, case in enumerate(plagiarism_cases):
        start = case.susp_off
        length = case.susp_len
        pred_intervals.append((start, length))
        colors.append(case_colors[idx % len(case_colors)])

    if pred_intervals:
        for i, (start, length) in enumerate(pred_intervals):
            ax.broken_barh([(start, length)], (0, 5), facecolors=colors[i])
            ax.text(start, 2.5, f"{start}", va='center', ha='right', fontsize=9)
            ax.text(start + length, 2.5, f"{start + length}", va='center', ha='left', fontsize=9)

    ax.set_ylim(-2, 18)
    ax.set_yticks([2.5, 12.5])
    ax.set_yticklabels(['Predicted', 'Ground Truth'])
    ax.set_xlabel('Suspicious Document Offset (Characters)')
    ax.set_title(f'Comparison for {susp_doc.doc_name}')
    ax.grid(True, axis='x', linestyle='--', alpha=0.3)
    plt.tight_layout()
    return fig

def run_demo_pipeline(suspicious_file, source_file, progress=gr.Progress()):
    t_start = time.time()
    progress(0.05, desc="Loading documents...")

    with open(suspicious_file.name, "r", encoding='utf-8', errors='ignore') as f:
        suspicious_text = f.read()
    with open(source_file.name, "r", encoding='utf-8', errors='ignore') as f:
        source_text = f.read()

    suspicious_doc = Document(doc_id=-1, is_source=False, text_content=suspicious_text)
    suspicious_doc.file_path = suspicious_file.name
    source_doc = Document(doc_id=-2, is_source=True, text_content=source_text)
    source_doc.file_path = source_file.name

    radar = WordEncoplotEngine()

    progress(0.25, desc="Selecting lexical anchors...")
    t_anchor = time.time()
    anchors = radar.get_anchors_for_pair(
        suspicious_doc, source_doc
    )
    time_anchor = time.time() - t_anchor

    semantic = SemanticAnalyzer(progress=progress)

    progress(0.40, desc="Running SBERT semantic verification...")
    t_sbert = time.time()
    verified_fragments, _, _ = semantic.verify_anchors(
        suspicious_doc, source_doc, anchors, progress=progress
    )
    time_sbert = time.time() - t_sbert

    progress(0.85, desc="Clustering results...")
    t_cluster = time.time()
    plagiarism_cases = cluster_results_dbscan(verified_fragments)
    time_cluster = time.time() - t_cluster

    progress(0.95, desc="Building visualization...")
    highlighted_html = build_highlighted_html(suspicious_text, plagiarism_cases)

    t_total = time.time() - t_start
    debug_data = {
        "time_anchors_s": round(time_anchor, 2),
        "num_anchors": len(anchors),
        "time_sbert_s": round(time_sbert, 2),
        "time_clustering_s": round(time_cluster, 2),
        "num_verified_fragments": len(verified_fragments),
        "num_final_cases": len(plagiarism_cases),
        "time_total_s": round(t_total, 2)
    }
    debug_json = json.dumps(debug_data, indent=4)
    df = build_case_dataframe(plagiarism_cases, suspicious_text, source_text)

    progress(1.0, desc="Done!")
    yield highlighted_html, df, debug_json

def run_multi_source_pipeline(suspicious_id, n_sources, progress=gr.Progress()):
    t_start = time.time()
    progress(0.0, desc="Extracting sources...")

    susp_doc, source_ids = extract_sources_for_suspicious(int(suspicious_id), int(n_sources))

    semantic = SemanticAnalyzer(progress=progress)
    radar = WordEncoplotEngine()

    time_hash_src = 0
    time_anchors = 0

    all_candidate_pairs = []

    for i, src_id in enumerate(source_ids):
        progress(0.15 + (0.25 * i / len(source_ids)), desc=f"Hashing & Anchors: Source {src_id} ({i+1}/{len(source_ids)})...")

        src_doc = Document(src_id, is_source=True)

        t1 = time.time()
        anchors = radar.get_anchors_for_pair(
            susp_doc, src_doc
        )
        time_anchors += time.time() - t1

        if anchors:
            all_candidate_pairs.append((src_doc, anchors))

    progress(0.45, desc="Running SBERT verification across all pairs...")
    t2 = time.time()
    verified, _, _ = semantic.verify_multiple(susp_doc, all_candidate_pairs, progress=progress)
    time_sbert = time.time() - t2

    progress(0.85, desc="Clustering results...")
    t3 = time.time()
    all_cases = []

    hits_by_source = defaultdict(list)
    for v in verified:
        hits_by_source[v.src_id].append(v)

    for src_id, hits in hits_by_source.items():
        cases = cluster_results_dbscan(hits)
        all_cases.extend(cases)

    time_cluster = time.time() - t3

    progress(0.9, desc="Building highlighted HTML...")
    highlighted = build_highlighted_html(susp_doc.text, all_cases)

    progress(0.95, desc="Building dataframe and plot...")
    df = build_case_dataframe(all_cases, susp_doc.text, None)
    fig = plot_validation_timeline(susp_doc, all_cases, CASE_COLORS)

    t_total = time.time() - t_start

    validator = ValidationMetrics()
    metrics = validator.calculate_metrics(susp_doc.plagiarism_features, all_cases)

    debug = {
        "suspicious_id": suspicious_id,
        "sources_used": len(source_ids),
        "time_anchors_s": round(time_anchors, 2),
        "time_sbert_s": round(time_sbert, 2),
        "time_clustering_s": round(time_cluster, 2),
        "cases_found": len(all_cases),
        "time_total_s": round(t_total, 2),
        "metrics": metrics
    }

    root = ET.Element("document", reference=susp_doc.doc_name)
    for c in all_cases:
        ET.SubElement(root, "feature",
            name="detected-plagiarism",
            this_offset=str(int(c.susp_off)),
            this_length=str(int(c.susp_len)),
            source_reference=f"source-document{c.src_id:05d}.txt",
            source_offset=str(int(c.src_off)),
            source_length=str(int(c.src_len))
        )
    xml_str = ET.tostring(root, encoding='utf-8')
    predicted_xml = minidom.parseString(xml_str).toprettyxml(indent="  ")

    try:
        with open(susp_doc.xml_path, 'r', encoding='utf-8') as f:
            gt_xml = f.read()
    except Exception:
        gt_xml = "<?xml version=\"1.0\" encoding=\"UTF-8\"?>\n<!-- Ground truth XML not found -->"

    progress(1.0, desc="Done!")
    yield highlighted, df, fig, json.dumps(debug, indent=2), predicted_xml, gt_xml


In [ ]:
with gr.Blocks() as demo:

    gr.Markdown("# Plagiarism Detection System")

    # 1vs1
    with gr.Tab("1 vs 1"):

        with gr.Row():
            suspicious_input = gr.File()
            source_input = gr.File()

        run_button = gr.Button("Run")

        with gr.Tabs():
            with gr.Tab("Highlighted Suspicious Document"):
                h1 = gr.HTML()
            with gr.Tab("Fragment Validation"):
                d1 = gr.Dataframe(wrap=True, interactive=False)
            with gr.Tab("Debug Information"):
                dbg1 = gr.Code(language="json")

    # 1vsmany
    with gr.Tab("1 vs Many"):

        pan_id = gr.Number(label="Suspicious ID", precision=0)
        pan_n = gr.Number(label="Sources", value=50)

        pan_btn = gr.Button("Run 1 vs Many")

        with gr.Tabs():
            with gr.Tab("Highlighted Suspicious Document"):
                h2 = gr.HTML()
            with gr.Tab("Fragment Validation"):
                timeline_plot = gr.Plot(label="Validation Timeline")
                d2 = gr.Dataframe(wrap=True, interactive=False)
            with gr.Tab("XML Comparison"):
                with gr.Row():
                    pred_xml_out = gr.Code(label="Predicted Output", language="html")
                    gt_xml_out = gr.Code(label="Ground Truth Output", language="html")
            with gr.Tab("Debug Information"):
                dbg2 = gr.Code(language="json")

    run_button.click(
        run_demo_pipeline,
        [suspicious_input, source_input],
        [h1, d1, dbg1],
        show_progress="none"
    )

    pan_btn.click(
        run_multi_source_pipeline,
        [pan_id, pan_n],
        [h2, d2, timeline_plot, dbg2, pred_xml_out, gt_xml_out],
        show_progress="none"
    )

demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://e26ba3cc7b8d6be3fa.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[SOURCES] Doc 00005 | GT sources: 1 | Target total: 50
[SOURCES] GT: [178] | Random fill: 49 | Total: 50
[INIT] Loading paraphrase-multilingual-mpnet-base-v2...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[INIT] Converting model to FP16 (Mixed Precision)
[SOURCES] Doc 00005 | GT sources: 1 | Target total: 100
[SOURCES] GT: [178] | Random fill: 99 | Total: 100
[INIT] Loading paraphrase-multilingual-mpnet-base-v2...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[INIT] Converting model to FP16 (Mixed Precision)
